# nb18 — ViDeBERTa load probe

Goal: load `Fsoft-AIC/videberta-base` **correctly** and find why the MRC benchmark scored it ~2 F1.

ViDeBERTa-base is a standard `deberta-v2` checkpoint (no custom code). Prime suspect = `DebertaV2TokenizerFast` `offset_mapping` misalignment → corrupted gold answer spans. The probe checks: (1) model loads, (2) encoder forward + fill-mask sane, (3) **gold span recoverable from offsets** (the real 2-F1 test), (4) raw vs PyVi word-seg fertility.

Run top-to-bottom. Read the **VERDICT** at the end.

In [1]:
!pip install -q transformers pyvi sentencepiece
# torch is preinstalled on Colab; salt3_videberta_load_probe.py must sit in the working dir

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 140.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 70.7 MB/s eta 0:00:00


In [3]:
from salt3_videberta_load_probe import run_all
result = run_all()  # prints VERDICT; result = {'forward_ok':..., 'span_ok':...}

=== ViDeBERTa load probe: Fsoft-AIC/videberta-base ===


config.json:   0%|          | 0.00/610 [00:00<?, ?B/s]

[1/load] model_type=deberta-v2  layers=12  hidden=768  vocab=128000


tokenizer_config.json:   0%|          | 0.00/401 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/8.49M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForQuestionAnswering LOAD REPORT from: Fsoft-AIC/videberta-base
Key                                        | Status     | 
-------------------------------------------+------------+-
mask_predictions.LayerNorm.weight          | UNEXPECTED | 
mask_predictions.dense.weight              | UNEXPECTED | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED | 
mask_predictions.classifier.bias           | UNEXPECTED | 
mask_predictions.classifier.weight         | UNEXPECTED | 
mask_predictions.dense.bias                | UNEXPECTED | 
mask_predictions.LayerNorm.bias            | UNEXPECTED | 
qa_outputs.weight                          | MISSING    | 
qa_outputs.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[1/load] AutoModelForQuestionAnswering loaded OK (so the 2-F1 is NOT a model-class bug)


model.safetensors:   0%|          | 0.00/567M [00:00<?, ?B/s]

[2/fwd] logits finite=True  top5 for [MASK]=['▁XI', 'CIS', '▁Dân_Cư', 'Đảm', '▁GS']
[3/offsets] context tokens=21  suspicious_offsets=0
[3/offsets] gold='Hà Nội'  recovered='Hà Nội'  (start_t=12, end_t=13)
[3/offsets] >>> SPAN RECOVERABLE: True 
[4/seg] RAW      (9 toks): ['▁Hà', '▁Nội', '▁là', '▁thủ', '▁đô', '▁của', '▁nước', '▁Việt', '▁Nam']
[4/seg] PyVi 'Hà_Nội là thủ_đô của nước Việt_Nam'
[4/seg] PyVi-seg (7 toks): ['▁Hà_Nội', '▁là', '▁', 'thủ_đô', '▁của', '▁nước', '▁Việt_Nam']
[4/seg] fertility raw=9 vs seg=7 (lower seg => word-level vocab actually used)

=== VERDICT ===
  model loads as deberta-v2 : True (not a load bug)
  encoder forward sane      : True
  gold-span recoverable     : True
  => Load+offsets fine. The 2 F1 lies elsewhere: check segmentation match (PyVi vs VnCoreNLP), add_prefix_space, or the QA fine-tune itself. Run a 5-example overfit test next.


## How to read it
- `forward_ok=True, span_ok=False` → **tokenizer offset bug**: the QA harness's char→token span mapping is corrupted for ViDeBERTa. Fix `build_qa_rows` to find the answer token-span by a token-id subsequence search (not `offset_mapping`), then re-benchmark. ViDeBERTa is fine.
- `forward_ok=True, span_ok=True` → loading+offsets are fine; the 2 F1 is from segmentation mismatch (PyVi vs VnCoreNLP) or the fine-tune. Add a 5-example overfit test.
- `forward_ok=False` → weights/checkpoint load problem (rare for deberta-v2).

Either way: if the encoder forward is sane, **ViDeBERTa-base is usable as a distillation teacher** (we only need hidden states / MLM logits, not the QA head).